# Car Sales — Modeling
**Team LGTSOW**
**Notebook 3 — Modeling**
**File:** `P1_CarSales-3_Modeling_LGTSOW.ipynb`

> **Note on execution:** This notebook was authored in an environment without internet
> access and without `xgboost`/`tensorflow` installed, so it could not be executed
> here. The code below is written to be run end-to-end in the team's conda
> environment (which already includes these packages). **Run this notebook
> top-to-bottom before committing**, so the outputs shown reflect real results, per
> the assignment's "fully executed, no errors" requirement.

> **Portability:** This notebook is written to run unmodified on either a laptop or
> the ACES cluster. It auto-detects available CPU cores, available memory, and GPU
> presence at runtime and sizes `GridSearchCV` parallelism / Keras device placement
> accordingly, instead of hardcoding numbers that only work for one machine. The one
> thing you may need to set by hand is where the data lives (see the data-directory
> cell below) if it isn't in the notebook's working directory on a given machine.

This notebook covers **Section 3** of the Notebook 3 spec: training the required
models (Section 2) and reporting their cross-validated and held-out performance in a
single comparison table (Section 3). Discussion (Section 4) and model export
(Section 5) are handled separately.

**ACES kernel path fix.** On ACES, `pip install --target=$SCRATCH/python_packages`
puts packages on disk, but this notebook's Jupyter kernel doesn't automatically look
there -- it isn't on `sys.path` by default, even though it works fine from an
interactive shell. This cell adds it explicitly, before anything tries to import a
package installed that way (`xgboost` in particular). Harmless (and a no-op) on a
laptop where `$SCRATCH` isn't set.

In [1]:
import sys, os

pkg_dir = os.path.expandvars('$SCRATCH/python_packages')
if os.path.isdir(pkg_dir) and pkg_dir not in sys.path:
    sys.path.insert(0, pkg_dir)
    print(f"Added to sys.path: {pkg_dir}")

Added to sys.path: /scratch/user/u.gd352312/python_packages


In [2]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

I0000 00:00:1789589716.819113 3324511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789589716.831158 3324511 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789589720.414452 3324511 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789589726.049884 3324511 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them 

## 0. Environment setup

**Data location.** The class data folder may live in a different place on your
laptop versus on ACES (a mapped drive, a home directory, a scratch/project path,
etc.). Rather than hardcoding one path, we check a short list of candidates and use
whichever one actually has `processed_train.csv` in it. If none of them work on your
machine, set the `CARSALES_DATA_DIR` environment variable (or just add your path to
`CANDIDATE_DATA_DIRS` below) instead of editing paths throughout the notebook.

In [3]:
import os

# Checked in order: an explicit override via the CARSALES_DATA_DIR environment
# variable, the notebook's own working directory (the common case on a laptop),
# then a couple of common ACES scratch/project directory conventions. Add your
# own path here if your course's ACES layout differs.
CANDIDATE_DATA_DIRS = [
    os.environ.get('CARSALES_DATA_DIR'),
    '.',
    os.path.expanduser('~/scratch/msds565'),
    os.path.expanduser('~/msds565/data'),
]

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if candidate and os.path.exists(os.path.join(candidate, 'processed_train.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find processed_train.csv in any candidate data directory: "
        f"{[c for c in CANDIDATE_DATA_DIRS if c]}\n"
        "Set the CARSALES_DATA_DIR environment variable to your class data folder, e.g.:\n"
        "  Windows (laptop):  set CARSALES_DATA_DIR=C:\\path\\to\\data\n"
        "  Linux / ACES:      export CARSALES_DATA_DIR=/path/to/data"
    )

print(f"Using data directory: {os.path.abspath(DATA_DIR)}")

Using data directory: /scratch/user/u.gd352312/Car_Sales


## 1. Setup

We read in `processed_train.csv` and `processed_test.csv` exactly as Notebook 2 saved
them — this is the 80/20, `random_state=42` split from preprocessing, and we don't
re-split or re-derive anything here.

In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'processed_train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed_test.csv'))

print(f"Train: {train_df.shape[0]:,} rows x {train_df.shape[1]} columns")
print(f"Test:  {test_df.shape[0]:,} rows x {test_df.shape[1]} columns")

Train: 241,847 rows x 278 columns
Test:  60,462 rows x 278 columns


**Building X and y.** `price` is the regression target. Notebook 2 also retained
the original object-dtype version of every one-hot-encoded categorical column
(`make_name`, `body_type`, `city`, etc.) specifically for the Notebook 5 fairness
audit — those columns must NOT be fed into the model here.

Rather than hardcoding that metadata column list a second time (and risking it
drifting out of sync with Notebook 2), we detect it directly: any column that's still
object/string-typed in the processed data is metadata, not a numeric model feature.
This also correctly catches `listed_date` (a raw date string) even though it isn't
one of the "fairness" metadata columns — it was never meant to be fed into a model
directly, since its useful signal was already extracted into `listing_month` /
`listing_dayofweek` during feature engineering.

The exact same column drops are applied to both the train and test splits.

In [5]:
metadata_cols = train_df.select_dtypes(include='object').columns.tolist()
print(f"Dropping {len(metadata_cols)} object-dtype column(s) from X (metadata retained for later, not used here):")
print(metadata_cols)

y_train = train_df['price']
X_train = train_df.drop(columns=['price'] + metadata_cols)

y_test = test_df['price']
X_test = test_df.drop(columns=['price'] + metadata_cols)

print()
print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

Dropping 16 object-dtype column(s) from X (metadata retained for later, not used here):
['meta_city', 'meta_bed', 'meta_body_type', 'meta_cabin', 'meta_exterior_color', 'meta_franchise_make', 'meta_fuel_type', 'meta_interior_color', 'listed_date', 'meta_listing_color', 'meta_make_name', 'meta_model_name', 'meta_transmission', 'meta_trim_name', 'meta_wheel_system', 'meta_engine_layout']

X_train: (241847, 261),  y_train: (241847,)
X_test:  (60462, 261),  y_test:  (60462,)


**Sizing `GridSearchCV` parallelism to the machine it's running on.**
`GridSearchCV`'s parallel workers are separate OS processes, and each one needs its
own copy of the training fold in memory -- so the right number of parallel workers
depends on both CPU core count *and* available RAM. On a shared HPC node like ACES,
naively trusting `os.cpu_count()` / `psutil`'s memory numbers is actually dangerous:
those reflect the **whole physical node**, not what your specific job was actually
allocated -- a shared node can report e.g. 96 cores and 500+ GB free even when your
job only has a small slice of that. Spawning a worker per *reported* core in that
situation can blow straight through your job's real memory limit and get silently
killed by the scheduler, with no Python traceback at all.

So we check SLURM's own allocation environment variables first (`SLURM_CPUS_PER_TASK`,
`SLURM_MEM_PER_NODE`/`SLURM_MEM_PER_CPU`) when they're present, since those reflect
what this job actually has -- falling back to the host-wide numbers only when SLURM
isn't in play (e.g. running this same notebook on a laptop). Either way, a hard
ceiling caps the result regardless, since a grid this size (a few dozen candidate x
fold fits per model) gets no benefit from dozens of parallel workers anyway -- the
per-process interpreter/library overhead just wastes memory past a certain point.

In [6]:
import os

try:
    import psutil
    available_bytes = psutil.virtual_memory().available
except ImportError:
    # psutil isn't installed -- fall back to a conservative fixed cap rather than
    # guessing at available memory. `conda install psutil` removes this fallback.
    available_bytes = None

# On a shared HPC node (ACES), os.cpu_count() and psutil's memory numbers reflect
# the WHOLE physical node, not what's actually allocated to this specific job --
# a Jupyter session can see e.g. 96 cores / 500+ GB of a shared node while its own
# SLURM allocation is a small fraction of that. Prefer SLURM's own environment
# variables when present, since those reflect the real per-job allocation; only
# fall back to the host-wide numbers (with a hard ceiling) when SLURM isn't in play,
# e.g. running this same notebook on a laptop.
slurm_cpus = os.environ.get('SLURM_CPUS_PER_TASK') or os.environ.get('SLURM_JOB_CPUS_PER_NODE')
if slurm_cpus is not None:
    cpu_count = int(slurm_cpus.split(',')[0])  # SLURM_JOB_CPUS_PER_NODE can be a comma list on multi-node jobs
    cpu_source = 'SLURM allocation'
else:
    cpu_count = os.cpu_count() or 1
    cpu_source = 'os.cpu_count() (whole machine -- not SLURM-aware)'

bytes_per_job = X_train.memory_usage(deep=True).sum()  # one copy of the full training matrix

slurm_mem_mb = os.environ.get('SLURM_MEM_PER_NODE') or os.environ.get('SLURM_MEM_PER_CPU')
if slurm_mem_mb is not None:
    # SLURM_MEM_PER_CPU needs multiplying by allocated CPUs; SLURM_MEM_PER_NODE is already the job total
    per_cpu = 'SLURM_MEM_PER_CPU' in os.environ
    available_bytes = int(slurm_mem_mb) * 1e6 * (cpu_count if per_cpu else 1)
    mem_source = 'SLURM allocation'
elif available_bytes is not None:
    mem_source = "psutil (whole machine -- not SLURM-aware, may overstate what's actually available to this job)"
else:
    mem_source = 'unavailable'

# Hard ceiling regardless of what the above computes -- protects against exactly the
# "psutil/os.cpu_count() says way more is available than this job actually has" trap
# that crashed the kernel earlier. Set this to roughly your allocated core count once
# you've confirmed (via the SLURM/psutil numbers printed below) that this job's real
# memory allocation can support it -- there's little point capping below your actual
# core count once memory genuinely isn't the constraint anymore.
HARD_CAP = 32

if available_bytes is not None:
    SAFETY_FRACTION = 0.5  # leave headroom for the main process, the OS, and (on ACES) other jobs sharing the node
    max_jobs_by_memory = max(1, int((available_bytes * SAFETY_FRACTION) // bytes_per_job))
else:
    max_jobs_by_memory = 2

N_JOBS = max(1, min(cpu_count, max_jobs_by_memory, HARD_CAP))

print(f"CPU count source:              {cpu_source}")
print(f"CPUs available (as counted):   {cpu_count}")
print(f"Est. memory per worker copy:   {bytes_per_job / 1e6:,.0f} MB")
print(f"Memory source:                 {mem_source}")
if available_bytes is not None:
    print(f"Available memory (as counted): {available_bytes / 1e9:.1f} GB")
print(f"Hard ceiling:                   {HARD_CAP}")
print(f"--> Using N_JOBS = {N_JOBS} for GridSearchCV")
print()
print("If this still runs out of memory (or crashes the kernel with no traceback --")
print("a sign of a job/cgroup memory limit being hit), lower HARD_CAP above and re-run.")

CPU count source:              SLURM allocation
CPUs available (as counted):   32
Est. memory per worker copy:   505 MB
Memory source:                 SLURM allocation
Available memory (as counted): 262.1 GB
Hard ceiling:                   32
--> Using N_JOBS = 32 for GridSearchCV

If this still runs out of memory (or crashes the kernel with no traceback --
a sign of a job/cgroup memory limit being hit), lower HARD_CAP above and re-run.


## 2. Models

### 2a. scikit-learn regressors

We train three regressor types — **Linear Regression** and **Random Forest** (both
required), plus **XGBoost** as the third. For each, `GridSearchCV` runs 5-fold CV on
the training set, scoring every fold on MAE, MAPE, and R² simultaneously so we can
report all three without re-fitting. `refit='R2'` tells `GridSearchCV` to pick the
best hyperparameter combination by mean CV R² and then refit that one configuration
on the *whole* training set — that refit model is `best_estimator_`, and
`refit_time_` gives us its training time for free.

A shared helper function keeps the three models' evaluation logic identical (same
metrics, same scoring dict, same reporting shape), so differences in the final table
reflect the models, not inconsistent evaluation code between them.

In [7]:
def run_gridsearch(name, estimator, param_grid, X_train, y_train, X_test, y_test, n_jobs=N_JOBS):
    """Run 5-fold GridSearchCV, then report CV-mean and held-out test metrics
    for the best (refit) estimator, plus its refit training time.

    n_jobs controls how many folds/candidates are fit in parallel. Each parallel
    worker is a separate OS process that needs its own copy of the training fold
    in memory, so a high n_jobs on a large dataset can exhaust RAM well before it
    exhausts CPU -- see the markdown note above for sizing guidance.
    """
    scoring = {
        'MAE': 'neg_mean_absolute_error',
        'MAPE': 'neg_mean_absolute_percentage_error',
        'R2': 'r2',
    }

    gs = GridSearchCV(
        estimator, param_grid,
        cv=5, scoring=scoring, refit='R2', n_jobs=n_jobs, pre_dispatch='n_jobs'
    )
    gs.fit(X_train, y_train)

    best_idx = gs.best_index_
    # neg_* scorers are stored negated (sklearn's "higher is better" convention) -- flip sign back
    cv_MAE_mean = -gs.cv_results_['mean_test_MAE'][best_idx]
    cv_MAPE_mean = -gs.cv_results_['mean_test_MAPE'][best_idx]
    cv_R2_mean = gs.cv_results_['mean_test_R2'][best_idx]

    # Which individual fold scored best for the winning hyperparameters, out of the 5
    # used in this GridSearchCV -- used later (Section 5) to number the exported file.
    per_fold_R2 = [gs.cv_results_[f'split{i}_test_R2'][best_idx] for i in range(gs.n_splits_)]
    best_fold = int(np.argmax(per_fold_R2)) + 1  # +1: folds are numbered 1-5, not 0-4

    test_preds = gs.best_estimator_.predict(X_test)
    test_MAE = mean_absolute_error(y_test, test_preds)
    test_MAPE = mean_absolute_percentage_error(y_test, test_preds)
    test_R2 = r2_score(y_test, test_preds)

    print(f"[{name}] best params: {gs.best_params_}")
    print(f"[{name}] CV  -> MAE={cv_MAE_mean:,.1f}  MAPE={cv_MAPE_mean:.3f}  R2={cv_R2_mean:.3f}")
    print(f"[{name}] Test-> MAE={test_MAE:,.1f}  MAPE={test_MAPE:.3f}  R2={test_R2:.3f}")
    print(f"[{name}] refit training time: {gs.refit_time_:.1f}s")
    print(f"[{name}] best-scoring fold: {best_fold} (of 5) -- used to number the exported model file")

    row = {
        'model': name,
        'cv_MAE_mean': cv_MAE_mean,
        'cv_MAPE_mean': cv_MAPE_mean,
        'cv_R2_mean': cv_R2_mean,
        'test_MAE': test_MAE,
        'test_MAPE': test_MAPE,
        'test_R2': test_R2,
        'training_time_sec': gs.refit_time_,
        'best_fold': best_fold,
    }
    return row, gs

**Linear Regression's parameter grid is intentionally small.** Plain OLS
(`LinearRegression`) doesn't have regularization hyperparameters to tune — the only
settings that actually change the fit are whether to fit an intercept and whether to
constrain coefficients to be non-negative. We grid-search both rather than padding
the grid artificially with settings that wouldn't change anything.

**Random Forest's grid** varies tree count, max depth, and minimum leaf size —
enough to meaningfully trade off underfitting (shallow, heavily-restricted trees)
against overfitting/runtime (deep, unrestricted trees) on ~242K rows.

**XGBoost's grid** varies tree count, max depth, and learning rate — the three
hyperparameters with the largest practical effect on a gradient-boosted tree
ensemble's bias/variance tradeoff.

In [8]:
linreg_grid = {
    'fit_intercept': [True, False],
    'positive': [False, True],
}
linreg_row, linreg_gs = run_gridsearch(
    'linreg', LinearRegression(), linreg_grid,
    X_train, y_train, X_test, y_test
)

[linreg] best params: {'fit_intercept': True, 'positive': False}
[linreg] CV  -> MAE=4,582.4  MAPE=0.227  R2=0.844
[linreg] Test-> MAE=4,597.1  MAPE=0.228  R2=0.842
[linreg] refit training time: 2.8s
[linreg] best-scoring fold: 3 (of 5) -- used to number the exported model file


In [9]:
rf_grid = {
    'n_estimators': [100, 300],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 4],
}
rf_row, rf_gs = run_gridsearch(
    'rf', RandomForestRegressor(random_state=42), rf_grid,
    X_train, y_train, X_test, y_test
)

[rf] best params: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}
[rf] CV  -> MAE=1,542.2  MAPE=0.068  R2=0.971
[rf] Test-> MAE=1,470.5  MAPE=0.065  R2=0.974
[rf] refit training time: 1572.7s
[rf] best-scoring fold: 3 (of 5) -- used to number the exported model file


In [10]:
xgb_grid = {
    'n_estimators': [100, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
}
xgb_row, xgb_gs = run_gridsearch(
    'xgb', XGBRegressor(random_state=42, objective='reg:squarederror'), xgb_grid,
    X_train, y_train, X_test, y_test
)

[xgb] best params: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 300}
[xgb] CV  -> MAE=1,605.5  MAPE=0.068  R2=0.974
[xgb] Test-> MAE=1,578.9  MAPE=0.067  R2=0.977
[xgb] refit training time: 24.6s
[xgb] best-scoring fold: 3 (of 5) -- used to number the exported model file


### 2b. Keras neural networks

We define three architectures that vary in depth, width, and regularization, and
treat them the way `GridSearchCV` treats hyperparameter combinations: run a manual
5-fold CV over the training set for each one, average the fold scores, and pick the
single best-performing architecture to retrain on the full training set. We do **not**
export the two architectures that lose — only the winner.

- **`nn1` (small):** two hidden layers (64 -> 32), no regularization. A lean baseline network.
- **`nn2` (medium, regularized):** three hidden layers (128 -> 64 -> 32) with dropout
  after the first two, to check whether a deeper network with explicit
  overfitting control does better than the plain baseline.
- **`nn3` (wide, batch-normalized):** two wide hidden layers (256 -> 128) with batch
  normalization, testing whether width (rather than depth) plus normalized
  activations helps more on this tabular data.

All three use Adam + MSE loss (standard for regression) and are evaluated on the same
MAE/MAPE/R² metrics as the sklearn models, so the final comparison table is apples-to-apples.

**GPU detection.** A laptop typically has no usable GPU for TensorFlow and will
train on CPU -- that's expected and fine given the network sizes below. ACES may
expose one or more GPUs depending on the node/kernel you've requested. Either way,
this cell just detects what's available and configures it; no manual edits are
needed to switch between the two. `set_memory_growth` is set specifically because
ACES GPUs may be shared across other people's jobs on the same node -- without it,
TensorFlow grabs all GPU memory up front by default, which can starve other jobs (or
your own next run) unnecessarily.

In [11]:
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"{len(gpus)} GPU(s) detected -- Keras will train on GPU:")
        for gpu in gpus:
            print(f"  {gpu.name}")
    except RuntimeError as e:
        # set_memory_growth must be called before GPUs are initialized; if this
        # notebook already ran a GPU op earlier, it's a no-op error, not a real problem
        print(f"Note: could not change GPU memory growth setting ({e})")
else:
    print("No GPU detected -- Keras will train on CPU.")
    print("This is expected on most laptops. On ACES, make sure you've requested a "
          "GPU-enabled node/kernel if you intended to train on GPU.")

No GPU detected -- Keras will train on CPU.
This is expected on most laptops. On ACES, make sure you've requested a GPU-enabled node/kernel if you intended to train on GPU.


E0000 00:00:1789593587.111817 3324511 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789593587.536165 3326343 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1789593588.017669 3324511 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789593588.017820 3326343 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789593588.049462 3324511 gpu_device.cc:2365] Cannot dlopen some GPU l

In [12]:
def build_nn1(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


def build_nn2(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


def build_nn3(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


architectures = {'nn1': build_nn1, 'nn2': build_nn2, 'nn3': build_nn3}

**Manual 5-fold CV.** We use the same `KFold(random_state=42)` splitting scheme
sklearn's `GridSearchCV` uses internally, so the CV folds here are constructed the
same way (though not literally the same row groupings as the sklearn models' folds,
since those were handled internally by `GridSearchCV`). A fresh model is built for
every fold — reusing one model instance across folds would leak learned weights from
earlier folds into later ones. `EarlyStopping` on validation loss prevents each fold
from training past the point of overfitting, and also keeps total runtime bounded.

In [13]:
X_train_arr = X_train.to_numpy()
y_train_arr = y_train.to_numpy()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

keras_cv_results = {}
keras_fold_r2_by_arch = {}

for name, build_fn in architectures.items():
    fold_mae, fold_mape, fold_r2 = [], [], []

    for fold_i, (tr_idx, val_idx) in enumerate(kf.split(X_train_arr), start=1):
        X_tr, X_val = X_train_arr[tr_idx], X_train_arr[val_idx]
        y_tr, y_val = y_train_arr[tr_idx], y_train_arr[val_idx]

        model = build_fn(X_train_arr.shape[1])
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        )
        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=50, batch_size=256,
            callbacks=[early_stop], verbose=0,
        )

        preds = model.predict(X_val, verbose=0).flatten()
        fold_mae.append(mean_absolute_error(y_val, preds))
        fold_mape.append(mean_absolute_percentage_error(y_val, preds))
        fold_r2.append(r2_score(y_val, preds))

        print(f"[{name}] fold {fold_i}: MAE={fold_mae[-1]:,.1f}  MAPE={fold_mape[-1]:.3f}  R2={fold_r2[-1]:.3f}")

    keras_cv_results[name] = {
        'cv_MAE_mean': np.mean(fold_mae),
        'cv_MAPE_mean': np.mean(fold_mape),
        'cv_R2_mean': np.mean(fold_r2),
        'cv_R2_std': np.std(fold_r2),   # checking fold-to-fold variability, per the assignment
    }
    keras_fold_r2_by_arch[name] = fold_r2   # kept for Section 5: which fold scored best for the winning architecture
    print(f"[{name}] CV mean R2 = {keras_cv_results[name]['cv_R2_mean']:.3f}  (std {keras_cv_results[name]['cv_R2_std']:.3f})")
    print()

keras_cv_summary = pd.DataFrame(keras_cv_results).T
keras_cv_summary

[nn1] fold 1: MAE=2,350.0  MAPE=0.096  R2=0.949
[nn1] fold 2: MAE=2,351.0  MAPE=0.096  R2=0.950
[nn1] fold 3: MAE=2,342.0  MAPE=0.095  R2=0.947
[nn1] fold 4: MAE=2,374.0  MAPE=0.096  R2=0.949
[nn1] fold 5: MAE=2,378.1  MAPE=0.096  R2=0.947
[nn1] CV mean R2 = 0.948  (std 0.001)

[nn2] fold 1: MAE=2,219.7  MAPE=0.097  R2=0.954
[nn2] fold 2: MAE=2,256.0  MAPE=0.094  R2=0.953
[nn2] fold 3: MAE=2,238.6  MAPE=0.094  R2=0.951
[nn2] fold 4: MAE=2,235.4  MAPE=0.094  R2=0.954
[nn2] fold 5: MAE=2,242.3  MAPE=0.096  R2=0.953
[nn2] CV mean R2 = 0.953  (std 0.001)

[nn3] fold 1: MAE=2,046.7  MAPE=0.088  R2=0.962
[nn3] fold 2: MAE=2,092.1  MAPE=0.082  R2=0.960
[nn3] fold 3: MAE=2,048.2  MAPE=0.086  R2=0.961
[nn3] fold 4: MAE=2,002.7  MAPE=0.082  R2=0.963
[nn3] fold 5: MAE=2,017.8  MAPE=0.083  R2=0.964
[nn3] CV mean R2 = 0.962  (std 0.001)



,cv_MAE_mean,cv_MAPE_mean,cv_R2_mean,cv_R2_std
nn1,2359.021489,0.095644,0.948349,0.001149
nn2,2238.387880,0.095029,0.953010,0.001076
nn3,2041.481006,0.084218,0.961962,0.001362


**Selecting and refitting the winning architecture.** We pick whichever
architecture had the highest mean CV R², then retrain *only* that one on the full
training set — the losing two architectures are discarded here, matching the
assignment's "export the winning architecture, not every one you tried" instruction.

In [14]:
best_arch_name = keras_cv_summary['cv_R2_mean'].idxmax()
print(f"Best architecture by mean CV R2: {best_arch_name}")

best_build_fn = architectures[best_arch_name]

start_time = time.time()
final_nn = best_build_fn(X_train_arr.shape[1])
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
final_nn.fit(
    X_train_arr, y_train_arr,
    validation_split=0.1,
    epochs=50, batch_size=256,
    callbacks=[early_stop], verbose=0,
)
nn_training_time = time.time() - start_time

nn_test_preds = final_nn.predict(X_test.to_numpy(), verbose=0).flatten()
nn_test_MAE = mean_absolute_error(y_test, nn_test_preds)
nn_test_MAPE = mean_absolute_percentage_error(y_test, nn_test_preds)
nn_test_R2 = r2_score(y_test, nn_test_preds)

# Which of the 5 manual CV folds scored best for the WINNING architecture specifically
# -- used in Section 5 to number the exported file, same convention as the sklearn models.
nn_best_fold = int(np.argmax(keras_fold_r2_by_arch[best_arch_name])) + 1

nn_row = {
    'model': best_arch_name,
    'cv_MAE_mean': keras_cv_summary.loc[best_arch_name, 'cv_MAE_mean'],
    'cv_MAPE_mean': keras_cv_summary.loc[best_arch_name, 'cv_MAPE_mean'],
    'cv_R2_mean': keras_cv_summary.loc[best_arch_name, 'cv_R2_mean'],
    'test_MAE': nn_test_MAE,
    'test_MAPE': nn_test_MAPE,
    'test_R2': nn_test_R2,
    'training_time_sec': nn_training_time,
    'best_fold': nn_best_fold,
}

print(f"[{best_arch_name}] Test -> MAE={nn_test_MAE:,.1f}  MAPE={nn_test_MAPE:.3f}  R2={nn_test_R2:.3f}")
print(f"[{best_arch_name}] refit training time: {nn_training_time:.1f}s")
print(f"[{best_arch_name}] best-scoring fold: {nn_best_fold} (of 5) -- used to number the exported model file")

Best architecture by mean CV R2: nn3
[nn3] Test -> MAE=1,962.7  MAPE=0.080  R2=0.964
[nn3] refit training time: 63.4s
[nn3] best-scoring fold: 5 (of 5) -- used to number the exported model file


## 3. Performance Table

One row per model — the three sklearn regressors (each already reduced to its single
best/refit configuration by `GridSearchCV`) plus the one winning Keras architecture
(already reduced from three candidates to the best one above). Every row reports the
same seven columns: mean CV MAE/MAPE/R² (from the training folds), the same three
metrics evaluated on the held-out `processed_test.csv`, and the wall-clock time to
refit that model on the whole training set.

In [15]:
performance_rows = [linreg_row, rf_row, xgb_row, nn_row]

model_metrics = pd.DataFrame(performance_rows).set_index('model')
model_metrics = model_metrics[[
    'cv_MAE_mean', 'cv_MAPE_mean', 'cv_R2_mean',
    'test_MAE', 'test_MAPE', 'test_R2',
    'training_time_sec',
]]
model_metrics

,cv_MAE_mean,cv_MAPE_mean,cv_R2_mean,test_MAE,test_MAPE,test_R2,training_time_sec
model,,,,,,,
linreg,4582.383364,0.227486,0.843945,4597.093078,0.228420,0.841928,2.799440
rf,1542.156152,0.068014,0.971051,1470.479823,0.065453,0.973958,1572.733973
xgb,1605.542833,0.068122,0.973958,1578.857217,0.067062,0.976568,24.620455
nn3,2041.481006,0.084218,0.961962,1962.663824,0.080302,0.964234,63.370022


**Reading this table:** CV and test-set metrics should be close to each other
for every row — a large gap (especially test R² noticeably lower than CV R²) is the
early-warning sign for either overfitting or, per the assignment's leakage warning, a
`price`-derived feature that snuck into `X`. `training_time_sec` is the cost of the
*single* refit on the whole training set for each model — not the total GridSearchCV
or manual-CV search time, which is considerably longer since it includes every
hyperparameter combination across all 5 folds.

## 4. Discussion

> **Note:** This section can't be filled in for real until the notebook is actually
> run in your conda environment -- the discussion below needs actual numbers from
> `model_metrics`, not guesses. The code cell right after this computes several
> diagnostic facts directly from your results (best/worst model, CV-vs-test gaps,
> training time) so you're not starting from a blank page. **Replace the bracketed
> placeholders in the markdown cell below with what your team actually observes**,
> and delete this note once you have.

In [16]:
# Diagnostic facts to ground the discussion below -- computed directly from
# model_metrics rather than eyeballed, so the write-up matches the real numbers.

best_model = model_metrics['test_R2'].idxmax()
worst_model = model_metrics['test_R2'].idxmin()
fastest_model = model_metrics['training_time_sec'].idxmin()
slowest_model = model_metrics['training_time_sec'].idxmax()

print(f"Best model (highest test R2):  {best_model}  (R2 = {model_metrics.loc[best_model, 'test_R2']:.3f})")
print(f"Worst model (lowest test R2):  {worst_model}  (R2 = {model_metrics.loc[worst_model, 'test_R2']:.3f})")
print(f"Fastest to train: {fastest_model}  ({model_metrics.loc[fastest_model, 'training_time_sec']:.1f}s)")
print(f"Slowest to train: {slowest_model}  ({model_metrics.loc[slowest_model, 'training_time_sec']:.1f}s)")
print()

# CV-vs-test gap: how much each model's held-out performance differs from its
# cross-validated performance. A large positive gap (CV R2 much higher than test R2)
# is the overfitting/leakage warning sign called out in the assignment.
gap = (model_metrics['cv_R2_mean'] - model_metrics['test_R2']).rename('cv_minus_test_R2')
print("CV R2 minus test R2 (positive = performed worse on held-out data than on CV folds):")
print(gap.sort_values(ascending=False).to_string())
print()

if model_metrics['test_R2'].max() > 0.98:
    print("WARNING: a test R2 above ~0.98 on a real-world price dataset is suspiciously")
    print("high -- re-check Notebook 2's feature engineering for anything derived from price.")

Best model (highest test R2):  xgb  (R2 = 0.977)
Worst model (lowest test R2):  linreg  (R2 = 0.842)
Fastest to train: linreg  (2.8s)
Slowest to train: rf  (1572.7s)

CV R2 minus test R2 (positive = performed worse on held-out data than on CV folds):
model
linreg    0.002018
nn3      -0.002271
xgb      -0.002609
rf       -0.002906



**Best-performing model: `xgb`, test R² = 0.977.** XGBoost edges out Random Forest
on R² (0.9766 vs. 0.9740) despite being trained with a much smaller, cheaper grid
search. Both tree ensembles substantially outperform Linear Regression (R² = 0.842)
and the winning neural network (`nn3`, R² = 0.964), which lines up with what the EDA
notebook found: `price` doesn't move linearly with the raw features (horsepower,
mileage, engine size) -- the relationships are non-linear and full of interactions
(e.g. mileage's effect on price depends heavily on vehicle age and segment), which is
exactly what tree-based models are built to capture and a plain linear model isn't.

**Worst-performing model: `linreg`, test R² = 0.842.** Still a reasonably strong
score in absolute terms, but its test MAPE (22.8%) is roughly 3.5x every other
model's (6.5-8.0%), meaning its typical price prediction is off by nearly a quarter
of the car's value. This is consistent with the multicollinearity the EDA notebook
already flagged among the size/power features (horsepower vs. engine displacement vs.
physical dimensions) -- a linear model has to split credit across correlated
predictors in a way that doesn't actually reflect how price responds to them jointly,
while the tree-based models can use whichever correlated feature is most useful at
each split without that penalty.

**CV-vs-test gap: essentially reassuring, in the "good" direction.** Every model's
test R² is *higher* than its mean CV R² (by 0.2-0.3 percentage points for `rf`,
`xgb`, and `nn3`; linreg is close to flat), not lower. That's not overfitting -- if
anything it's the expected effect of the final refit: `GridSearchCV`'s refit trains
on the *entire* training set, while each individual CV fold only ever sees 80% of
it, so the refit model has strictly more data to learn from than what its own CV
score was measured on. None of the four models comes close to the ~0.98 threshold
that would raise a `price`-leakage concern.

**Training time vs. performance trade-off is the most interesting result here.**
`linreg` is fastest (2.8s) but also weakest by a wide margin -- expected, and not a
real trade-off worth taking given how much accuracy it gives up. The real trade-off
is `xgb` (24.6s) vs. `rf` (1,572.7s, ~26 minutes): XGBoost trains **~64x faster** than
Random Forest while achieving a *higher* test R² (0.9766 vs. 0.9740) -- though Random
Forest actually has a lower test MAE (\$1,470 vs. \$1,579), meaning it's slightly
more accurate on a typical prediction even though it's worse on the R²/squared-error
metric. That split suggests XGBoost is doing comparatively better on the largest
errors/outliers (which R² penalizes heavily) while Random Forest is marginally
tighter on the bulk of "normal" predictions. Given XGBoost is competitive-to-better
on every metric at a fraction of the training cost, it's the clear practical choice
between the two -- Random Forest's ~26-minute training time bought essentially nothing
here.

**Keras vs. sklearn:** the winning architecture, `nn3` (the wide, batch-normalized
network), reached R² = 0.964 -- solidly ahead of Linear Regression, but behind both
tree ensembles, and at more than double XGBoost's training time (63.4s vs. 24.6s) for
a worse result. This tracks with a common pattern on tabular data like this: gradient-
boosted trees and Random Forests tend to do well on structured/tabular data with lots
of one-hot-encoded categorical features right out of the box, while a feedforward
network usually needs more extensive architecture search, feature-specific
preprocessing (e.g. embeddings for the high-cardinality categoricals instead of raw
one-hot vectors), and more tuning than the 3 candidate architectures tried here to
close that gap. Given the training-time cost was already higher than XGBoost's for a
worse result, the added complexity of the neural network isn't earning its keep on
this dataset.

**Bottom line:** XGBoost is the best model here by nearly every measure that matters
(highest test R², second-lowest MAE, by far the fastest of the three high-accuracy
models to train), and would be the natural pick to carry forward into feature
selection and interpretability if we could only keep one.

## 5. Export

**Saving the performance table.** `model_metrics` is written to the class data
folder as `model_metrics.csv` -- Notebook 4 reads this back in for its
before/after comparison.

**Saving the models.** Each sklearn model's `best_estimator_` (the one
`GridSearchCV` already refit on the whole training set) is pickled, and the
winning Keras architecture is saved via `model.save(...)`. File names follow
`P1_N<num>_<modeltype>.pkl` / `.keras` -- `<num>` is that specific model's own
best-scoring fold out of its 5 CV folds (computed back in Section 2 and carried
in each row's `best_fold`), so it can differ from model to model.

In [17]:
import pickle

MODELS_DIR = 'models'   # matches the models/ folder at the repo root
os.makedirs(MODELS_DIR, exist_ok=True)

# Save the performance table to the class data folder (Notebook 4 reads this back in)
metrics_path = os.path.join(DATA_DIR, 'model_metrics.csv')
model_metrics.to_csv(metrics_path)
print(f"Saved: {metrics_path}")

Saved: ./model_metrics.csv


In [18]:
# sklearn models: pickle each best_estimator_, named by that model's own best CV fold
sklearn_exports = {
    'linreg': (linreg_gs.best_estimator_, linreg_row['best_fold']),
    'rf': (rf_gs.best_estimator_, rf_row['best_fold']),
    'xgb': (xgb_gs.best_estimator_, xgb_row['best_fold']),
}

for model_type, (estimator, best_fold) in sklearn_exports.items():
    filename = f"P1_N{best_fold}_{model_type}.pkl"
    filepath = os.path.join(MODELS_DIR, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(estimator, f)
    print(f"Saved: {filepath}")

Saved: models/P1_N3_linreg.pkl
Saved: models/P1_N3_rf.pkl
Saved: models/P1_N3_xgb.pkl


In [19]:
# Winning Keras architecture: same naming convention, .keras extension
keras_filename = f"P1_N{nn_best_fold}_{best_arch_name}.keras"
keras_filepath = os.path.join(MODELS_DIR, keras_filename)
final_nn.save(keras_filepath)
print(f"Saved: {keras_filepath}")

Saved: models/P1_N5_nn3.keras
